In [1]:
import os
os.environ.setdefault('http_proxy', 'http://webproxy.au.harveynorman.com:8080')
os.environ.setdefault('https_proxy', 'http://webproxy.au.harveynorman.com:8080')

'http://webproxy.au.harveynorman.com:8080'

## Partitioned Queries in Athena

```sql
-- Create partitioned folder using Athena
CREATE EXTERNAL TABLE myretail.orders_part (
    order_id INT,
    order_date STRING,
    order_customer_id INT,
    order_status STRING
) PARTITIONED BY (order_month INT)
STORED AS parquet
LOCATION 's3://romadv-itv-retail/myretail/order_part'

-- develop query for partitioned table
SELECT o.*,
    replace(substring(order_date, 1, 7), '-', '') AS order_month
FROM myretail.orders AS o
LIMIT 10;

-- insert into partitioned tables using athena
INSERT INTO myretail.orders_part
SELECT o.*,
    cast(replace(substring(order_date, 1, 7), '-', '') AS INT) AS order_month
FROM myretail.orders AS o

-- validate data partitioning
DESCRIBE FORMATTED myretail.orders_part

SHOW PARTITIONS myretail.orders_part

SELECT * FROM myretail.orders_part LIMIT 10;

SELECT COUNT(*) FROM myretail.orders_part;

-- drop athena tables and delete data files
SELECT * FROM myretail.order_items LIMIT 10;

DROP TABLE myretail.order_items
-- only metadata about order_items is dropped but
-- the data in the bucket will remain, to fully remove
-- delete the files in the s3 bucket location too

SELECT * FROM myretail.orders_part LIMIT 10;
DROP TABLE myretail.orders_part

-- data partitioning using CTAS
CREATE TABLE myretail.orders_part
WITH (
    format = 'TEXTFILE',
    external_location = 's3://romadv-itv-retail/myretail/orders_part/',
    field_delimiter = ',',
    partitioned_by = ARRAY['order_month']
)
AS
SELECT 
    o.*,
    cast(replace(substring(order_date, 1, 7), '-', '') AS INT) AS order_month
FROM myretail.orders AS o;
```

## Using Athena in AWS CLI
```bash
# get help for athena
aws athena help

```

In [5]:
%%sh
# list dbs in awsdatacatalog 
aws athena list-databases --catalog-name AwsDataCatalog --profile itvadmin

{
    "DatabaseList": [
        {
            "Name": "default"
        },
        {
            "Name": "flights-db",
            "Description": "For ITversity Udemy course flights data "
        },
        {
            "Name": "itvghlandingdb"
        },
        {
            "Name": "itvghrawdb"
        },
        {
            "Name": "myretail"
        },
        {
            "Name": "retail_db"
        }
    ]
}


In [7]:
%%sh
aws athena list-work-groups \
    --profile itvadmin

{
    "WorkGroups": [
        {
            "Name": "primary",
            "State": "ENABLED",
            "Description": "",
            "CreationTime": "2025-10-13T16:29:37.257000+11:00",
            "EngineVersion": {
                "SelectedEngineVersion": "AUTO",
                "EffectiveEngineVersion": "Athena engine version 3"
            }
        }
    ]
}


In [8]:
%%sh
aws athena get-work-group \
    --work-group primary \
    --profile itvadmin \
    --region ap-southeast-2

{
    "WorkGroup": {
        "Name": "primary",
        "State": "ENABLED",
        "Configuration": {
            "ResultConfiguration": {},
            "EnforceWorkGroupConfiguration": false,
            "PublishCloudWatchMetricsEnabled": true,
            "RequesterPaysEnabled": false,
            "EngineVersion": {
                "SelectedEngineVersion": "AUTO",
                "EffectiveEngineVersion": "Athena engine version 3"
            },
            "EnableMinimumEncryptionConfiguration": false
        },
        "CreationTime": "2025-10-13T16:29:37.257000+11:00"
    }
}


In [9]:
%%sh
aws athena start-query-execution --query-string "SELECT COUNT(1) FROM myretail.orders" \
    --profile itvadmin \
    --region ap-southeast-2
# query will fail because ResultConfiguration or location of query results has not been set


An error occurred (InvalidRequestException) when calling the StartQueryExecution operation: No output location provided. You did not provide an output location for  your query results. Either specify an S3 bucket location or enable Athena managed query results in your workgroup settings.


CalledProcessError: Command 'b'aws athena start-query-execution --query-string "SELECT COUNT(1) FROM myretail.orders" \\\n    --profile itvadmin \\\n    --region ap-southeast-2\n'' returned non-zero exit status 254.

In [10]:
%%sh
# added a result location for the work group in aws console
aws athena start-query-execution --query-string "SELECT COUNT(1) FROM myretail.orders" \
    --profile itvadmin \
    --region ap-southeast-2

{
    "QueryExecutionId": "893099db-d492-403e-95b8-73cec935a324"
}


In [ ]:
%%sh
aws athena start-query-execution help

In [ ]:
%%sh
aws athena get-query-execution help
# Capture query execution id from query above

In [3]:
%%sh
aws athena get-query-execution \
--query-execution-id 893099db-d492-403e-95b8-73cec935a324 \
--profile itvadmin \
--region ap-southeast-2

{
    "QueryExecution": {
        "QueryExecutionId": "893099db-d492-403e-95b8-73cec935a324",
        "Query": "SELECT COUNT(1) FROM myretail.orders",
        "StatementType": "DML",
        "ResultConfiguration": {
            "OutputLocation": "s3://romadv-itvathena/wgprimary/893099db-d492-403e-95b8-73cec935a324.csv"
        },
        "ResultReuseConfiguration": {
            "ResultReuseByAgeConfiguration": {
                "Enabled": false
            }
        },
        "QueryExecutionContext": {},
        "Status": {
            "State": "SUCCEEDED",
            "SubmissionDateTime": "2026-01-08T22:14:32.658000+11:00",
            "CompletionDateTime": "2026-01-08T22:14:33.412000+11:00"
        },
        "Statistics": {
            "EngineExecutionTimeInMillis": 613,
            "DataScannedInBytes": 584990,
            "TotalExecutionTimeInMillis": 754,
            "QueryQueueTimeInMillis": 57,
            "ServicePreProcessingTimeInMillis": 54,
            "QueryPlanningTim

In [6]:
%%sh
aws s3 cp s3://romadv-itvathena/wgprimary/893099db-d492-403e-95b8-73cec935a324.csv . --profile itvadmin

download: s3://romadv-itvathena/wgprimary/893099db-d492-403e-95b8-73cec935a324.csv to ./893099db-d492-403e-95b8-73cec935a324.csv


In [10]:
%%sh
tail ./893099db-d492-403e-95b8-73cec935a324.csv

"_col0"
"68883"


## Get Athena Table Metadata using AWS CLI

In [ ]:
%%sh
aws athena list-table-metadata help

In [12]:
%%sh
aws athena list-table-metadata \
    --catalog-name AwsDataCatalog \
    --database-name myretail \
    --region ap-southeast-2 \
    --profile itvadmin

{
    "TableMetadataList": [
        {
            "Name": "orders",
            "CreateTime": "2026-01-08T17:15:30+11:00",
            "LastAccessTime": "1970-01-01T10:00:00+10:00",
            "TableType": "EXTERNAL_TABLE",
            "Columns": [
                {
                    "Name": "order_id",
                    "Type": "int",
                    "Comment": "from deserializer"
                },
                {
                    "Name": "order_date",
                    "Type": "string",
                    "Comment": "from deserializer"
                },
                {
                    "Name": "order_customer_id",
                    "Type": "int",
                    "Comment": "from deserializer"
                },
                {
                    "Name": "order_status",
                    "Type": "string",
                    "Comment": "from deserializer"
                }
            ],
            "PartitionKeys": [],
            "Parameters": {
 

In [ ]:
%%sh
aws athena get-table-metadata help

In [14]:
%%sh
aws athena get-table-metadata \
    --catalog-name AwsDataCatalog \
    --database-name myretail \
    --table-name orders_part \
    --region ap-southeast-2 \
    --profile itvadmin

{
    "TableMetadata": {
        "Name": "orders_part",
        "CreateTime": "2026-01-08T21:35:30+11:00",
        "TableType": "EXTERNAL_TABLE",
        "Columns": [
            {
                "Name": "order_id",
                "Type": "int"
            },
            {
                "Name": "order_date",
                "Type": "string"
            },
            {
                "Name": "order_customer_id",
                "Type": "int"
            },
            {
                "Name": "order_status",
                "Type": "string"
            }
        ],
        "PartitionKeys": [
            {
                "Name": "order_month",
                "Type": "int"
            }
        ],
        "Parameters": {
            "EXTERNAL": "TRUE",
            "auto.purge": "false",
            "has_encrypted_data": "false",
            "inputformat": "org.apache.hadoop.mapred.TextInputFormat",
            "location": "s3://romadv-itv-retail/myretail/orders_part/",
          

In [ ]:
!aws athena start-query-execution help

In [18]:
!aws s3 ls s3://romadv-itv-retail/myretail/ --profile itvadmin

                           PRE orders/
                           PRE orders_part/


In [20]:
!aws athena start-query-execution \
    --query-string "SELECT count(*) FROM myretail.orders" \
    --result-configuration OutputLocation=s3://romadv-itv-retail/myretail/order_count_awscli \
    --region ap-southeast-2 \
    --profile itvadmin

{
    "QueryExecutionId": "02220b7d-d004-4447-9c1d-cdb225c4acf2"
}


In [21]:
!aws athena get-query-execution \
    --query-execution-id 02220b7d-d004-4447-9c1d-cdb225c4acf2 \
    --region ap-southeast-2 \
    --profile itvadmin

{
    "QueryExecution": {
        "QueryExecutionId": "02220b7d-d004-4447-9c1d-cdb225c4acf2",
        "Query": "SELECT count(*) FROM myretail.orders",
        "StatementType": "DML",
        "ResultConfiguration": {
            "OutputLocation": "s3://romadv-itv-retail/myretail/order_count_awscli/02220b7d-d004-4447-9c1d-cdb225c4acf2.csv"
        },
        "ResultReuseConfiguration": {
            "ResultReuseByAgeConfiguration": {
                "Enabled": false
            }
        },
        "QueryExecutionContext": {},
        "Status": {
            "State": "SUCCEEDED",
            "SubmissionDateTime": "2026-01-09T14:38:35.990000+11:00",
            "CompletionDateTime": "2026-01-09T14:38:36.694000+11:00"
        },
        "Statistics": {
            "EngineExecutionTimeInMillis": 525,
            "DataScannedInBytes": 584990,
            "TotalExecutionTimeInMillis": 704,
            "QueryQueueTimeInMillis": 103,
            "ServicePreProcessingTimeInMillis": 55,
         

In [22]:
!aws s3 ls s3://romadv-itv-retail/myretail/order_count_awscli/ --profile itvadmin

2026-01-09 14:38:37         16 02220b7d-d004-4447-9c1d-cdb225c4acf2.csv
2026-01-09 14:38:37         67 02220b7d-d004-4447-9c1d-cdb225c4acf2.csv.metadata


## Drop Athena table using AWS CLI

In [23]:
!aws athena start-query-execution \
    --query-string "DROP TABLE myretail.order_items" \
    --region ap-southeast-2 \
    --profile itvadmin

{
    "QueryExecutionId": "dd6392ad-1ae8-4b39-b0a0-b77c90adb269"
}


In [24]:
!aws athena get-query-execution \
    --query-execution-id dd6392ad-1ae8-4b39-b0a0-b77c90adb269 \
    --region ap-southeast-2 \
    --profile itvadmin

{
    "QueryExecution": {
        "QueryExecutionId": "dd6392ad-1ae8-4b39-b0a0-b77c90adb269",
        "Query": "DROP TABLE myretail.order_items",
        "StatementType": "DDL",
        "ResultConfiguration": {
            "OutputLocation": "s3://romadv-itvathena/wgprimary/dd6392ad-1ae8-4b39-b0a0-b77c90adb269.txt"
        },
        "ResultReuseConfiguration": {
            "ResultReuseByAgeConfiguration": {
                "Enabled": false
            }
        },
        "QueryExecutionContext": {},
        "Status": {
            "State": "SUCCEEDED",
            "SubmissionDateTime": "2026-01-09T15:02:50.552000+11:00",
            "CompletionDateTime": "2026-01-09T15:02:51.149000+11:00"
        },
        "Statistics": {
            "EngineExecutionTimeInMillis": 398,
            "DataScannedInBytes": 0,
            "TotalExecutionTimeInMillis": 597,
            "QueryQueueTimeInMillis": 163,
            "ServicePreProcessingTimeInMillis": 13,
            "ServiceProcessingTimeInMi

In [26]:
!aws s3 ls s3://romadv-itv-retail/myretail/order_items --profile itvadmin

                           PRE order_items/


In [28]:
!aws s3 rm s3://romadv-itv-retail/myretail/order_items/ --recursive --profile itvadmin

delete: s3://romadv-itv-retail/myretail/order_items/20260109_040158_00043_u8epz_1c2a6ecb-6443-4e61-8697-5c597974398c.gz


In [29]:
!aws s3 ls s3://romadv-itv-retail/myretail/ --profile itvadmin

                           PRE order_count_awscli/
                           PRE orders/
                           PRE orders_part/


## Run CTAS under Athena using AWS CLI

In [30]:
!aws athena start-query-execution \
    --query-string "CREATE TABLE myretail.order_items \
        WITH ( \
            format = 'TEXTFILE', \
            external_location = 's3://romadv-itv-retail/myretail/order_items/', \
            field_delimiter = ',' \
        ) \
        AS \
        SELECT * FROM retail_db.order_items" \
    --region ap-southeast-2 \
    --profile itvadmin

{
    "QueryExecutionId": "10bf68e6-74c1-4f37-a721-a0a64db8d721"
}


In [31]:
!aws athena get-query-execution \
    --query-execution-id 10bf68e6-74c1-4f37-a721-a0a64db8d721 \
    --region ap-southeast-2 \
    --profile itvadmin

{
    "QueryExecution": {
        "QueryExecutionId": "10bf68e6-74c1-4f37-a721-a0a64db8d721",
        "Query": "CREATE TABLE myretail.order_items          WITH (              format = 'TEXTFILE',              external_location = 's3://romadv-itv-retail/myretail/order_items/',              field_delimiter = ','          )          AS          SELECT * FROM retail_db.order_items",
        "StatementType": "DDL",
        "ResultConfiguration": {
            "OutputLocation": "s3://romadv-itvathena/wgprimary/tables/10bf68e6-74c1-4f37-a721-a0a64db8d721"
        },
        "ResultReuseConfiguration": {
            "ResultReuseByAgeConfiguration": {
                "Enabled": false
            }
        },
        "QueryExecutionContext": {},
        "Status": {
            "State": "SUCCEEDED",
            "SubmissionDateTime": "2026-01-09T15:09:49.841000+11:00",
            "CompletionDateTime": "2026-01-09T15:09:51.653000+11:00"
        },
        "Statistics": {
            "EngineExecuti

In [32]:
!aws s3 ls s3://romadv-itv-retail/myretail/order_items/ --profile itvadmin

2026-01-09 15:09:52    1030031 20260109_040949_00115_u8wvc_ce47dac8-46cb-427c-ae81-db437b6c6fd7.gz
